# Regridding-resolution sensitivity of EERIE `tas` biases (1° vs 0.25°)

**Question.** When we evaluate a high-resolution EERIE model against ERA5, how much does the
*target regridding resolution* change the climatological `tas` bias we report? We regrid each
model (and ERA5) onto a **1°** grid and onto a **0.25°** grid, form the bias `model − obs` on each,
and inspect where the two disagree — then compare against the coarse **CMIP6** benchmark.

The hypothesis: over **mountain ranges** and **coastlines** — where the surface-temperature
field has steep gradients — coarsening to 1° smears structure that 0.25° keeps, so the reported
bias is resolution-sensitive there. CMIP6 (native ~1–2°) cannot resolve it at all, so the
EERIE-vs-CMIP6 gap should be largest in exactly those regions.

**Workflow**
1. Load EERIE `tas` monthly → annual climatology (1980–2014); regrid to 1° and 0.25°.
2. Same for ERA5; bias = model − obs on each grid.
3. Sensitivity = (0.25° bias coarsened to 1°) − (1° bias), plus the within-1°-cell spread of the
   0.25° bias — both localise where resolution matters.
4. Auto-rank the largest-difference land regions, then compare a curated set of mountain /
   coastal boxes across EERIE@1°, EERIE@0.25° and CMIP6 MMM.

> **Where to run.** Loading full-resolution EERIE `tas` and regridding is heavy — run this on a
> DKRZ **compute node** (SLURM/Jupyter), not the login node. Activate the `feather` conda env.

In [ ]:
# Keep BLAS from spawning a thread per core (avoids RLIMIT_NPROC errors on DKRZ).
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import nereus as nr

from feather.config import FeatherConfig
from feather.data.cmor_loader import CMORLoader
from feather.data.obs import ObsLoader
from feather.data.cmip6 import CMIP6Loader
from feather.util.temporal import climatology, seasonal_climatology

plt.rcParams["figure.dpi"] = 110

## 1. Configuration & loaders

We reuse `configs/eerie.yaml` (EERIE Ensemble + CMIP6 benchmark). Everything is `tas`, annual
climatology, over the config period.

In [ ]:
CONFIG_PATH = "../configs/eerie.yaml"
VAR = "tas"
RES_HI, RES_LO = 0.25, 1.0            # target regridding resolutions (deg)
COARSEN = int(round(RES_LO / RES_HI)) # 0.25 -> 1.0 block factor (=4)

cfg = FeatherConfig.from_yaml(CONFIG_PATH)
PERIOD = cfg.get_period()
INFL = float(cfg.nereus.get("influence_radius", 80_000.0))

model_loader = CMORLoader(cfg)
obs_loader = ObsLoader(cfg)
cmip6_loader = CMIP6Loader(cfg) if cfg.cmip6.get("enabled", False) else None

EERIE_MODELS = list(cfg.models)
print("period :", PERIOD)
print("EERIE  :", EERIE_MODELS)
print("CMIP6  :", list(cmip6_loader.models) if cmip6_loader else "(disabled)")

## 2. Regridding helper

One nereus call per (field, resolution). We use `method="linear"` and the config influence
radius, on a 0..360 longitude frame, and return an `xr.DataArray(lat, lon)`. The two target
resolutions are built from the *same* native field, so any difference is purely the target grid.

In [ ]:
def regrid_field(field, lon, lat, resolution, influence_radius=INFL, method="linear"):
    """Regrid a scattered/rectilinear field to a regular `resolution`-deg grid."""
    vals = np.asarray(field).ravel()
    lon = np.asarray(lon)
    lat = np.asarray(lat)
    if lon.ndim == 1 and lon.shape[0] != vals.shape[0]:
        lon, lat = np.meshgrid(lon, lat)
    _, interp = nr.regrid(
        vals, lon=np.asarray(lon), lat=np.asarray(lat),
        resolution=resolution, method=method,
        influence_radius=influence_radius,
        lon_bounds=(0.0, 360.0), as_xarray=True,
    )
    tlat = np.asarray(interp.target_lat)[:, 0]
    tlon = np.asarray(interp.target_lon)[0, :]
    return xr.DataArray(
        interp(vals), dims=("lat", "lon"),
        coords={"lat": tlat, "lon": tlon}, name=VAR,
    )


def to_common_1deg(bias_hi, bias_lo):
    """Block-mean the 0.25° bias to the 1° grid and align it with the 1° bias."""
    coarse = bias_hi.coarsen(lat=COARSEN, lon=COARSEN, boundary="trim").mean()
    coarse = coarse.interp(lat=bias_lo.lat, lon=bias_lo.lon)  # snap to 1° centres
    return coarse


def within_cell_std(bias_hi):
    """Std of the 0.25° bias inside each 1° cell — structure that 1° hides."""
    return bias_hi.coarsen(lat=COARSEN, lon=COARSEN, boundary="trim").std()


def season_clim(da, period, season):
    """2-D climatology for ``'annual'``, ``'DJF'``, ``'MAM'``, ``'JJA'`` or ``'SON'``."""
    if season == "annual":
        return climatology(da, period).compute()
    return seasonal_climatology(da, period)[season].compute()

## 3. ERA5 reference climatology on both grids

In [ ]:
obs_da = obs_loader.load_for_model_var(VAR, PERIOD)          # ERA5 t2m [K]
obs_clim = climatology(obs_da, PERIOD).compute()
obs_lat = obs_clim["lat" if "lat" in obs_clim.coords else "latitude"].values
obs_lon = obs_clim["lon" if "lon" in obs_clim.coords else "longitude"].values

obs_hi = regrid_field(obs_clim.values, obs_lon, obs_lat, RES_HI)
obs_lo = regrid_field(obs_clim.values, obs_lon, obs_lat, RES_LO)
print("ERA5 regridded:", dict(hi=obs_hi.shape, lo=obs_lo.shape))

## 4. EERIE model climatologies & biases on both grids

For each EERIE model we store the bias at 1°, the 0.25° bias coarsened to 1°, their difference
(the *sensitivity*), and the within-cell std.

In [ ]:
eerie = {}  # model -> dict of fields on the 1° grid
for m in EERIE_MODELS:
    try:
        da = model_loader.load_var(m, VAR, period=PERIOD, time_mean=False)
    except (KeyError, FileNotFoundError) as e:
        print(f"  skip {m}: {e}")
        continue
    clim = climatology(da, PERIOD).compute()
    mlat = clim["lat"].values
    mlon = clim["lon"].values
    bias_hi = regrid_field(clim.values, mlon, mlat, RES_HI) - obs_hi
    bias_lo = regrid_field(clim.values, mlon, mlat, RES_LO) - obs_lo
    hi_on_lo = to_common_1deg(bias_hi, bias_lo)
    eerie[m] = {
        "bias_lo": bias_lo,
        "bias_hi_on_lo": hi_on_lo,
        "sensitivity": hi_on_lo - bias_lo,
        "cell_std": within_cell_std(bias_hi).interp(lat=bias_lo.lat, lon=bias_lo.lon),
    }
    print(f"  {m}: done")
print("models with data:", list(eerie))

## 5. Sensitivity map (one model)

Left: the 1° bias. Middle: 0.25° bias (coarsened to 1°). Right: their difference — the
regridding sensitivity. Coastlines overlaid; the difference lights up along orography and coasts.

In [ ]:
show = next(iter(eerie))
d = eerie[show]
fields = [("1° bias", d["bias_lo"], "RdBu_r", 6),
          ("0.25° bias (→ 1°)", d["bias_hi_on_lo"], "RdBu_r", 6),
          ("sensitivity (0.25° − 1°)", d["sensitivity"], "PuOr_r", 2)]
fig, axes = plt.subplots(1, 3, figsize=(18, 4.2),
                         subplot_kw={"projection": ccrs.Robinson()})
for ax, (ttl, fld, cmap, vm) in zip(axes, fields):
    p = ax.pcolormesh(fld.lon, fld.lat, fld, cmap=cmap, vmin=-vm, vmax=vm,
                      shading="auto", transform=ccrs.PlateCarree())
    ax.coastlines(linewidth=0.4)
    ax.set_title(f"{show} — {ttl}", fontsize=10)
    fig.colorbar(p, ax=ax, orientation="horizontal", pad=0.04, shrink=0.85,
                 label="K")
fig.suptitle(f"tas bias vs ERA5 — regridding-resolution sensitivity ({PERIOD[0]}–{PERIOD[1]})",
             y=1.03)
plt.show()

## 6. Auto-identify the largest-difference regions

Average `|sensitivity|` across EERIE models (land only), then list the top 1° cells. This tells
us *objectively* where the target resolution matters most before we pick named regions.

In [ ]:
# Land mask on the 1° grid from cartopy's land geometry.
import shapely.ops as sops
from shapely.geometry import Point
from shapely.prepared import prep
land_geom = prep(sops.unary_union(list(cfeature.LAND.geometries())))

sens_stack = xr.concat([v["sensitivity"] for v in eerie.values()], dim="model")
abs_sens = np.abs(sens_stack).mean("model")

lon2d, lat2d = np.meshgrid(abs_sens.lon.values, abs_sens.lat.values)
lon_pm = ((lon2d + 180) % 360) - 180
land = np.array([land_geom.contains(Point(x, y))
                 for x, y in zip(lon_pm.ravel(), lat2d.ravel())]).reshape(lat2d.shape)
abs_sens_land = abs_sens.where(land)

flat = abs_sens_land.stack(cell=("lat", "lon")).dropna("cell")
top = flat.sortby(flat, ascending=False)[:15]
print("Top land cells by mean |sensitivity| (K):")
for c in top.cell.values:
    la, lo = c
    print(f"  lat {la:6.1f}  lon {lo:6.1f}   {float(top.sel(cell=c)):.3f} K")

In [ ]:
fig = plt.figure(figsize=(12, 5))
ax = plt.axes(projection=ccrs.Robinson())
p = ax.pcolormesh(abs_sens_land.lon, abs_sens_land.lat, abs_sens_land,
                  cmap="magma_r", vmin=0, vmax=float(abs_sens_land.quantile(0.99)),
                  shading="auto", transform=ccrs.PlateCarree())
ax.coastlines(linewidth=0.4)
ax.set_title("Mean |0.25° − 1° tas-bias| across EERIE models (land)", fontsize=11)
fig.colorbar(p, ax=ax, orientation="vertical", shrink=0.8, label="K")
plt.show()

## 7. Curated mountain & coastal regions

Bounding boxes (lon in 0..360). Mountains: Alps, Tibetan Plateau, Andes, Rockies. Coasts:
Norwegian, US West, Chilean, New Zealand. Adjust from the auto-ranked cells above.

In [ ]:
REGIONS = {
    # name              (lat_min, lat_max, lon_min, lon_max, kind)
    "Alps":            (43, 48,   5,  16, "mountain"),
    "Tibetan Plateau": (27, 40,  75, 100, "mountain"),
    "Andes":           (-40, -18, 286, 296, "mountain"),
    "Rockies":         (35, 49,  244, 254, "mountain"),
    "Norwegian coast": (58, 68,   4,  14, "coastal"),
    "US West coast":   (34, 48,  234, 240, "coastal"),
    "Chilean coast":   (-40, -25, 284, 290, "coastal"),
    "New Zealand":     (-47, -34, 166, 179, "coastal"),
}

def box_mean(field, box):
    """cos(lat)-weighted mean over a lat/lon box; NaN cells are ignored."""
    lat0, lat1, lon0, lon1, _ = box
    sub = field.sel(lat=slice(lat0, lat1), lon=slice(lon0, lon1))
    w = np.cos(np.deg2rad(sub.lat))          # broadcasts over lon
    return float(sub.weighted(w).mean().values)

## 8. CMIP6 benchmark bias (MMM) on both grids

Regrid every configured CMIP6 model individually to each target grid, then average — the
“regrid-then-average” MMM convention used throughout Feather. CMIP6 is coarse, so its 1° and
0.25° renderings are near-identical; we keep the 1° one as the benchmark column.

In [ ]:
cmip6_bias_lo = None
if cmip6_loader is not None:
    members = []
    for cm in cmip6_loader.models:
        da = cmip6_loader.load_var("tas", cm, table="Amon", period=PERIOD, time_mean=True)
        if da is None:
            continue
        clat = da["lat"].values if "lat" in da.coords else da["latitude"].values
        clon = da["lon"].values if "lon" in da.coords else da["longitude"].values
        try:
            reg = regrid_field(da.values, clon, clat, RES_LO,
                               influence_radius=max(INFL, 250_000.0))
        except Exception as e:  # noqa: BLE001
            print(f"  skip CMIP6 {cm}: {e}")
            continue
        members.append(reg - obs_lo)
    if members:
        cmip6_bias_lo = xr.concat(members, dim="model").mean("model")
        print(f"CMIP6 MMM from {len(members)} models")
print("CMIP6 benchmark available:", cmip6_bias_lo is not None)

## 9. Regional comparison table & bars

Per region: EERIE ensemble-mean bias at 1°, at 0.25°, the resolution shift (0.25°−1°), and the
CMIP6 MMM bias. Mountains/coasts should show the biggest resolution shift and the biggest
EERIE-vs-CMIP6 gap.

In [ ]:
import pandas as pd

ens_lo = xr.concat([v["bias_lo"] for v in eerie.values()], dim="model").mean("model")
ens_hi = xr.concat([v["bias_hi_on_lo"] for v in eerie.values()], dim="model").mean("model")

rows = []
for name, box in REGIONS.items():
    b_lo = box_mean(ens_lo, box)
    b_hi = box_mean(ens_hi, box)
    row = {"region": name, "kind": box[4],
           "EERIE 1°": b_lo, "EERIE 0.25°": b_hi,
           "shift (0.25−1°)": b_hi - b_lo}
    row["CMIP6 MMM"] = box_mean(cmip6_bias_lo, box) if cmip6_bias_lo is not None else np.nan
    row["|EERIE0.25−CMIP6|"] = abs(b_hi - row["CMIP6 MMM"])
    rows.append(row)

df = pd.DataFrame(rows).set_index("region").round(3)
df

In [ ]:
x = np.arange(len(df))
w = 0.27
fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - w, df["EERIE 1°"], w, label="EERIE 1°", color="#8da0cb")
ax.bar(x, df["EERIE 0.25°"], w, label="EERIE 0.25°", color="#1f77b4")
ax.bar(x + w, df["CMIP6 MMM"], w, label="CMIP6 MMM", color="#999999")
ax.axhline(0, color="k", lw=0.7)
ax.set_xticks(x)
ax.set_xticklabels([f"{n}\n({k})" for n, k in zip(df.index, df["kind"])],
                   rotation=30, ha="right", fontsize=9)
ax.set_ylabel("tas bias vs ERA5 [K]")
ax.set_title("Regional tas bias: regridding resolution vs CMIP6 benchmark")
ax.legend()
fig.tight_layout()
plt.show()

## 10. Seasonal variant (DJF & JJA)

The regridding sensitivity is season-dependent: snow-line and coastal temperature gradients are
sharpest in the cold season, so **DJF** typically shows the strongest 1°-vs-0.25° differences over
mid-latitude mountains, while **JJA** dominates where summer heat contrasts are steep. `season_clim`
(defined in §2) picks the season; `compute_bias_set` reruns the whole §3–§8 pipeline for one season
and returns the 1°-grid fields. Reuses the `land` mask (§6), `REGIONS` and `box_mean` (§7).

> This recomputes model climatologies per season, so it roughly doubles the runtime of the annual
> pass. Trim `EERIE_MODELS` or `SEASON_SETS` for a quick look.

In [ ]:
def compute_bias_set(season):
    """All 1°-grid bias fields for one season (reuses the §2 helpers)."""
    o = season_clim(obs_da, PERIOD, season)
    olat = o["lat" if "lat" in o.coords else "latitude"].values
    olon = o["lon" if "lon" in o.coords else "longitude"].values
    o_hi = regrid_field(o.values, olon, olat, RES_HI)
    o_lo = regrid_field(o.values, olon, olat, RES_LO)

    ee = {}
    for m in EERIE_MODELS:
        try:
            da = model_loader.load_var(m, VAR, period=PERIOD, time_mean=False)
        except (KeyError, FileNotFoundError):
            continue
        c = season_clim(da, PERIOD, season)
        b_hi = regrid_field(c.values, c["lon"].values, c["lat"].values, RES_HI) - o_hi
        b_lo = regrid_field(c.values, c["lon"].values, c["lat"].values, RES_LO) - o_lo
        hol = to_common_1deg(b_hi, b_lo)
        ee[m] = {"bias_lo": b_lo, "bias_hi_on_lo": hol, "sensitivity": hol - b_lo}

    c6 = None
    if cmip6_loader is not None:
        mem = []
        for cm in cmip6_loader.models:
            da = cmip6_loader.load_var(
                "tas", cm, table="Amon", period=PERIOD,
                season=None if season == "annual" else season, time_mean=True,
            )
            if da is None:
                continue
            clat = da["lat"].values if "lat" in da.coords else da["latitude"].values
            clon = da["lon"].values if "lon" in da.coords else da["longitude"].values
            try:
                reg = regrid_field(da.values, clon, clat, RES_LO,
                                   influence_radius=max(INFL, 250_000.0))
            except Exception:  # noqa: BLE001
                continue
            mem.append(reg - o_lo)
        if mem:
            c6 = xr.concat(mem, dim="model").mean("model")

    ens_lo = xr.concat([v["bias_lo"] for v in ee.values()], dim="model").mean("model")
    ens_hi = xr.concat([v["bias_hi_on_lo"] for v in ee.values()], dim="model").mean("model")
    abs_sens = np.abs(
        xr.concat([v["sensitivity"] for v in ee.values()], dim="model")
    ).mean("model")
    return {"eerie": ee, "cmip6": c6, "ens_lo": ens_lo,
            "ens_hi": ens_hi, "abs_sens": abs_sens}


SEASON_SETS = {s: compute_bias_set(s) for s in ["DJF", "JJA"]}
print("computed seasons:", list(SEASON_SETS))

In [ ]:
fig, axes = plt.subplots(1, len(SEASON_SETS), figsize=(7 * len(SEASON_SETS), 4.2),
                         subplot_kw={"projection": ccrs.Robinson()})
axes = np.atleast_1d(axes)
for ax, (s, S) in zip(axes, SEASON_SETS.items()):
    fld = S["abs_sens"].where(land)          # `land` mask from §6
    p = ax.pcolormesh(fld.lon, fld.lat, fld, cmap="magma_r",
                      vmin=0, vmax=float(fld.quantile(0.99)),
                      shading="auto", transform=ccrs.PlateCarree())
    ax.coastlines(linewidth=0.4)
    ax.set_title(f"{s}: mean |0.25° − 1° tas-bias| (land)", fontsize=10)
    fig.colorbar(p, ax=ax, orientation="horizontal", pad=0.04, shrink=0.85, label="K")
fig.suptitle("Seasonal regridding-resolution sensitivity", y=1.02)
plt.show()

In [ ]:
import pandas as pd

seas_rows = []
for s, S in SEASON_SETS.items():
    for name, box in REGIONS.items():
        b_lo = box_mean(S["ens_lo"], box)
        b_hi = box_mean(S["ens_hi"], box)
        c6 = box_mean(S["cmip6"], box) if S["cmip6"] is not None else np.nan
        seas_rows.append({
            "season": s, "region": name, "kind": box[4],
            "EERIE 1°": b_lo, "EERIE 0.25°": b_hi,
            "shift (0.25−1°)": b_hi - b_lo, "CMIP6 MMM": c6,
            "|EERIE0.25−CMIP6|": abs(b_hi - c6),
        })

df_seas = pd.DataFrame(seas_rows).set_index(["season", "region"]).round(3)
df_seas

## 11. Takeaways

- The **sensitivity map** (§5) and the **auto-ranked cells** (§6) should concentrate along major
  orography (Himalaya/Tibet, Andes, Rockies, Alps, Greenland rim) and steep coasts — confirming
  that the 1°-vs-0.25° choice matters precisely where sub-grid gradients are strong.
- The **regional table** (§9): large `shift (0.25−1°)` and large `|EERIE0.25−CMIP6|` in mountain /
  coastal boxes quantifies the added value of resolving these gradients.
- The **seasonal maps/table** (§10) show *when* it matters: expect DJF to dominate over
  mid-latitude mountains (snow line / cold-season gradients) and JJA where summer heat contrasts
  are steepest. Compare `shift (0.25−1°)` across DJF vs JJA per region.
- If EERIE@1° and CMIP6 converge while EERIE@0.25° diverges, the divergence is genuine
  fine-scale signal, not a regridding artefact.

*Next steps:* add other variables (`pr`), or promote this into a registered
`GridResolutionSensitivity` diagnostic if it proves useful.